# Conditional Expectation (node 27)

Interactive companion to `README.md` and `lesson.tex`. We illustrate $E[X \mid Y]$ and the tower property using two fair dice, then peek at the $L^2$-projection viewpoint with a small linear-regression example.

**Setup.** $Y$ = first die, $W$ = second die, $X = Y + W$. Analytically, $E[X \mid Y = y] = y + 3.5$ and $E[X] = 7$.

Stdlib only.

## 1. Simulate the joint distribution

We draw many independent $(Y, W)$ pairs and bin by the value of $Y$.

In [ ]:
import random
from collections import defaultdict

rng = random.Random(0)
n = 200_000
sum_by_y = defaultdict(float)
count_by_y = defaultdict(int)
total_x = 0.0
for _ in range(n):
    y = rng.randint(1, 6)
    w = rng.randint(1, 6)
    x = y + w
    sum_by_y[y] += x
    count_by_y[y] += 1
    total_x += x
print(f'drew {n:,} pairs')


## 2. Empirical conditional means $E[X \mid Y = y]$

For each observed value of $Y$, average $X$ across the rows that match.

In [ ]:
cond_means = {y: sum_by_y[y] / count_by_y[y] for y in sorted(count_by_y)}
print(f"{'y':>3} {'E[X|Y=y] sim':>14} {'y+3.5':>8} {'abs err':>10}")
for y, m in cond_means.items():
    print(f'{y:>3} {m:>14.4f} {y+3.5:>8.2f} {abs(m-(y+3.5)):>10.4f}')


## 3. Tower property

$E[E[X \mid Y]] = \sum_y E[X \mid Y = y]\, P(Y = y)$ should equal $E[X]$.  
Because we are averaging the *same* sample, this identity holds **exactly** on the sample, not just approximately.

In [ ]:
p_y = {y: count_by_y[y] / n for y in cond_means}
tower = sum(cond_means[y] * p_y[y] for y in cond_means)
ex_hat = total_x / n
print(f'E[E[X|Y]] (via cond means) = {tower:.6f}')
print(f'E[X]      (sample mean)    = {ex_hat:.6f}')
print(f'analytic  E[X]             = 7.000000')
print(f'|tower - sample mean|      = {abs(tower-ex_hat):.2e}')


## 4. The projection viewpoint

$E[X \mid Y]$ minimises $E[(X - f(Y))^2]$ over functions $f$.  For our dice, the minimiser is $f(y) = y + 3.5$.  
Below we sweep over candidate affine predictors $f(y) = a\, y + b$ and check that the MSE is minimised near $(a, b) = (1, 3.5)$.

In [ ]:
import itertools
samples = []
rng2 = random.Random(1)
for _ in range(50_000):
    y = rng2.randint(1, 6); w = rng2.randint(1, 6)
    samples.append((y, y + w))

def mse(a, b):
    return sum((x - (a*y + b))**2 for y, x in samples) / len(samples)

grid = [(a, b) for a in [0.5, 0.9, 1.0, 1.1, 1.5] for b in [2.5, 3.0, 3.5, 4.0, 4.5]]
best = min(grid, key=lambda ab: mse(*ab))
print(f'best (a, b) on grid: {best}    MSE = {mse(*best):.4f}')
print(f'MSE at analytic minimiser (1, 3.5) = {mse(1, 3.5):.4f}')


## 5. Conditioning on a coarser $\sigma$-algebra

Let $\mathcal{H} = \sigma(\mathbf{1}\{Y \le 3\})$ — the smallest $\sigma$-algebra that knows only whether the first die is low ($\le 3$) or high ($\ge 4$). Then $E[X \mid \mathcal{H}]$ takes just two values.

In [ ]:
low_sum, low_n, high_sum, high_n = 0.0, 0, 0.0, 0
for y in cond_means:
    if y <= 3:
        low_sum += sum_by_y[y]; low_n += count_by_y[y]
    else:
        high_sum += sum_by_y[y]; high_n += count_by_y[y]
low_mean = low_sum / low_n
high_mean = high_sum / high_n
print(f'E[X | Y <= 3] (sim) = {low_mean:.4f}     analytic = {2 + 3.5}')   # y in {1,2,3}, mean y = 2
print(f'E[X | Y >= 4] (sim) = {high_mean:.4f}    analytic = {5 + 3.5}')   # y in {4,5,6}, mean y = 5
# Tower H ⊂ σ(Y): averaging the finer cond. means within each block reproduces these.
low_tower  = sum(cond_means[y] * (count_by_y[y]/low_n)  for y in [1,2,3])
high_tower = sum(cond_means[y] * (count_by_y[y]/high_n) for y in [4,5,6])
print(f'tower-built E[X | Y<=3] = {low_tower:.4f}, E[X | Y>=4] = {high_tower:.4f}')


## 6. Recap

- $E[X \mid Y]$ is a random variable (a function of $Y$), with values that match the analytic $y + 3.5$.
- The tower property $E[E[X \mid Y]] = E[X]$ is *exact* on the sample, not just asymptotic.
- Among affine predictors of $X$ from $Y$, the one minimising MSE is the conditional expectation $y + 3.5$.
- Conditioning on a coarser $\sigma$-algebra collapses neighbouring $y$-values; the tower property links coarse and fine conditionings.

Next: see `lesson.tex` for the Radon–Nikodym existence proof and the full tower-property proof.